In [10]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import time

options = webdriver.ChromeOptions()
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
options.add_argument("--start-maximized")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
ranking_data = []

try:
    # ==============================================================================
    #    시작
    # ==============================================================================

    # 1. 메인 페이지 접속
    print(" 1. 풍투데이 메인 페이지 접속 중")
    driver.get("https://poong.today/")
    time.sleep(3) 

    # 2. 상단의 '숲 방송 랭킹' 클릭
    print(" 2. '숲 방송 랭킹' 메뉴를 누릅니다")
    soop_btn = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, "//*[contains(text(), '숲 방송 랭킹') or contains(text(), 'SOOP')]"))
    )
    driver.execute_script("arguments[0].click();", soop_btn) # 방해물 무시 강제 클릭
    time.sleep(2)

    # 3. 기간 선택에서 '지난 달' 클릭 (드롭다운 아님 버튼 바로 클릭)
    print(" 3. 기간을 '지난 달'로 변경합니다")
    # 이미지에 보이는 대로 '지난 달' (띄어쓰기 포함) 버튼을 바로 찾아서 누릅니다.
    last_month_btn = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, "//*[text()='지난 달' or text()='지난달']"))
    )
    driver.execute_script("arguments[0].click();", last_month_btn)
    time.sleep(2) # 화면 데이터가 지난달로 바뀔 때까지 대기

    # 4. 카테고리 선택에서 드롭다운 열어서 '버추얼' 선택
    print(" 4. 카테고리에서 '버추얼'을 선택합니다...")
    # 이제 화면에 있는 드롭다운 상자(.react-dropdown-select-content)는 카테고리 하나뿐이니 바로 누릅니다
    category_box = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, ".react-dropdown-select-content"))
    )
    driver.execute_script("arguments[0].click();", category_box)
    time.sleep(1) # 드롭다운 스르륵 열리는 거 대기
    
    virtual_btn = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, "//*[text()='버추얼']"))
    )
    driver.execute_script("arguments[0].click();", virtual_btn)
    time.sleep(2) # 버추얼 데이터로 싹 바뀔 때까지 대기

    # 5. 페이지 빈 공간 아무 데나 눌러서 드롭다운 닫기 (이거안하면 꺼짐)
    print(" 5. 드롭다운을 치웁니다")
    # 키보드 'ESC' 키를 눌러서 열려있는 팝업이나 드롭다운을 강제로 닫는 가장 확실한 방법
    ActionChains(driver).send_keys(Keys.ESCAPE).perform()
    time.sleep(1)

    # 6. 하단으로 스크롤 후 더보기(+50위) 클릭
    print(" 6. 맨 밑으로 내려가서 '더보기' 버튼을 누릅니다")
    try:
        more_btn = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.XPATH, "//*[contains(text(), '더보기') or contains(text(), '+50')]"))
        )
        # 스크롤 내리기
        driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", more_btn)
        time.sleep(1) 
        
        # 강제 클릭 (앞에 뭐가 가려져 있어도 무조건 누름)
        driver.execute_script("arguments[0].click();", more_btn)
        print(" '더보기' 쫙 폈습니다")
        time.sleep(3) 
    except Exception as e:
        print("  더보기 버튼이 없거나 이미 열려있네요. 그냥 진행합니다")

    # ==============================================================================
    # 7.  탑 100을 수집
    # ==============================================================================
    print(" 7. 탑 100 데이터 수집 시작")

    # tbody>tr 대신 화면에 있는 모든 리스트(ul > li)를 싹 다 긁어옵니다.
    rows = driver.find_elements(By.CSS_SELECTOR, "ul > li") 
    
    print(f" 화면에서 총 {len(rows)}개의 리스트 줄을 발견했습니다. 랭킹 데이터만 골라냅니다...\n")

    for i, row in enumerate(rows):
        try:
            # 1. 줄 안의 전체 글자를 가져와서 엔터(\n) 기준으로 쪼갭니다.
            raw_text = row.text.strip()
            if not raw_text: continue # 빈 줄은 패스
                
            data = raw_text.split('\n')

            #  핵심 필터링: 쪼갠 데이터가 6개 이상이고, 첫 번째 값이 '숫자(순위)'인 경우만 랭킹 데이터로 인정
            # (이러면 상단 메뉴바나 하단 광고 리스트는 자동으로 무시됩니다)
            if len(data) >= 6 and data[0].isdigit():
                
                # 1등 데이터가 어떻게 쪼개졌는지 파이썬 콘솔에서 눈으로 한 번 확인
                if data[0] == "1":
                    print(f" 1등 원본 텍스트 쪼개기 결과: {data}")

                # 2. 데이터 매핑 
                # (중간에 'Live', '버추얼' 등 태그 개수가 달라도 무조건 배열의 맨 뒤에서부터 찾습니다)
                rank = data[0]               # 첫 번째는 무조건 순위
                bj_name = data[1]            # 두 번째는 무조건 이름

                broadcasting_time = data[-1] # 맨 마지막은 무조건 방송시간
                cumulative_viewers = data[-2].replace(",", "") # 뒤에서 2번째: 누적시청자수
                viewers = data[-3].replace(",", "")            # 뒤에서 3번째: 시청자수
                hourly_rate = data[-4].replace(",", "")        # 뒤에서 4번째: 시급(풍)
                total_poong = data[-5].replace(",", "")        # 뒤에서 5번째: 별풍선총합

                item = {
                    '순위': rank,
                    'BJ이름': bj_name,
                    '별풍선총합': total_poong,
                    '시급(풍)': hourly_rate,
                    '시청자수': viewers,
                    '누적시청자수': cumulative_viewers,
                    '방송시간': broadcasting_time
                }
                ranking_data.append(item)
                
                # 진척도 확인
                if rank in ["1", "50", "100"]:
                    print(f" {rank}등 데이터 캡처 성공: {bj_name}")
            
        except Exception as e:
            pass

except Exception as e:
    print("\n에러 발생 ㅠ_ㅠ:", e)
    
finally:
    print("\n 브라우저를 닫습니다.")
    driver.quit()

# CSV 저장 (pandas)
if ranking_data:
    df = pd.DataFrame(ranking_data)
    for col in ['별풍선총합', '시급(풍)', '시청자수', '누적시청자수']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print("\n 추출된 데이터 미리보기:")
    print(df.head())

    filename = "virtual_poong_last_month_top100_v5.csv"
    df.to_csv(filename, index=False, encoding='utf-8-sig') 
    print(f"\n '{filename}' 파일 완성")
else:
    print("\n 데이터가 없어요...")

 1. 풍투데이 메인 페이지 접속 중
 2. '숲 방송 랭킹' 메뉴를 누릅니다
 3. 기간을 '지난 달'로 변경합니다
 4. 카테고리에서 '버추얼'을 선택합니다...
 5. 드롭다운을 치웁니다
 6. 맨 밑으로 내려가서 '더보기' 버튼을 누릅니다
 '더보기' 쫙 폈습니다
 7. 탑 100 데이터 수집 시작
 화면에서 총 128개의 리스트 줄을 발견했습니다. 랭킹 데이터만 골라냅니다...

 1등 원본 텍스트 쪼개기 결과: ['1', '릴파♬', '버추얼', '813,222', '6,847', '20,013', '1,885,297', '4일 22 시간']
 1등 데이터 캡처 성공: 릴파♬
 50등 데이터 캡처 성공: 마야100
 100등 데이터 캡처 성공: 새히

 브라우저를 닫습니다.

 추출된 데이터 미리보기:
  순위  BJ이름   별풍선총합  시급(풍)   시청자수   누적시청자수       방송시간
0  1   릴파♬  813222   6847  20013  1885297   4일 22 시간
1  2    누야  736938   8046     38     7910   3일 19 시간
2  3  하루아이  603779   2423     47    19164   10일 9 시간
3  4  연초록♪  558739   3423   1398   419958   6일 19 시간
4  5  킹냥이_  437098   1555    733   288394  11일 17 시간

 'virtual_poong_last_month_top100_v5.csv' 파일 완성
